# 02 — Spatial canopy disagreement and crown merging diagnostics

**Repository:** `kvanoost/Musanga` · branch `v2` · `Crown/02_SAM1_Spatial_Error_Diagnostics.ipynb`  
**Run:** Google Colab **CPU** · **No SAM inference** · existing six-ROI outputs only.

**Purpose:** Understand why ROI 3 overpredicts canopy cover, ROI 4 underpredicts it, and ROI 1 agrees well despite large crown-density and crown-size biases.

- Area-based canopy **intersection, SAM-only, reference-only**, IoU, precision, recall, and percentage-point cover bias for **all six ROIs**, for author-style and NMS predictions.
- Spatial RGB overlays emphasizing **ROIs 1, 3, 4**.
- A **distinct crown-association diagnostic** for missing, merged, fragmented and unmatched crowns, based on polygon overlap; these are candidate mechanisms, not validated object-detection ground truth.
- Reproducible figures and CSVs in a new folder `V2/Crown/spatial_diagnostics/`. All input predictions and annotations are read-only.

**Scientific caution:** Kristof's inventory is provisional for full canopy coverage. Area-based agreement is agreement **with that inventory**, not necessarily ecological ground truth. The human-reference area cutoff is 5 m²; SAM polygons were generated with a 2–2500 m² area window.

## 01 — Input paths, configuration and software

In [ ]:
# 01 — CPU setup and exact six-ROI data paths
!pip -q install geopandas rasterio shapely scipy

from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from shapely.geometry import Polygon, MultiPolygon, box
from shapely.ops import unary_union
from shapely import make_valid
from scipy.optimize import linear_sum_assignment
from google.colab import drive
from rasterio.windows import from_bounds, Window

drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
DATA = ROOT / "data/Geo_Ref"
RGB = DATA / "Baego_georef.tif"
ROI_FILE = DATA / "Annotations/ROI_Crowns_Kvo.shp"
KRISTOF_FILE = DATA / "Annotations/Crown_Kristof.shp"
PRED_ROOT = ROOT / "V2/Crown/sam1_core_centroid_halo40_v2"
OUT = ROOT / "V2/Crown/spatial_diagnostics"
ROI_IDS = tuple(range(6))
METHODS = ("author_style", "nms_060")
MAP_ROIS = (1, 3, 4)
REF_MIN_AREA_M2 = 5.0
MATCH_IOU = 0.50
ASSOCIATION_REF_COVERAGE = 0.25   # heuristic: fraction of a reference polygon covered by one SAM polygon

for p in (RGB, ROI_FILE, KRISTOF_FILE, PRED_ROOT):
    if not p.exists():
        raise FileNotFoundError(p)
for roi in ROI_IDS:
    for method in METHODS:
        p = PRED_ROOT / f"roi{roi}_{method}.gpkg"
        if not p.is_file():
            raise FileNotFoundError(f"Missing consistent six-ROI output: {p}")
OUT.mkdir(parents=True, exist_ok=True)
with rasterio.open(RGB) as ds:
    CRS, BOUNDS = ds.crs, ds.bounds
assert CRS is not None and CRS.is_projected, "Projected metre-based CRS is required"
print("ROI outputs:", PRED_ROOT)
print("New diagnostic results:", OUT)
print("Raster CRS:", CRS)

## 02 — Load reference crowns and predicted polygons

Repair geometries **in memory**, preserve the missing CRS convention used by Notebook 00, and load all six matching SAM1 datasets. Whole-file SAM polygon counts include the 40 m halo.

In [ ]:
# 02 — Consistent projected geospatial inputs
def polygonal(g):
    if g is None or g.is_empty:
        return Polygon()
    if not g.is_valid:
        g = make_valid(g)
    if isinstance(g, (Polygon, MultiPolygon)):
        return g
    parts = [x for x in getattr(g, "geoms", [])
             if isinstance(x, (Polygon, MultiPolygon)) and not x.is_empty]
    return unary_union(parts) if parts else Polygon()

def load_polygons(path):
    g = gpd.read_file(path)
    g = g.loc[g.geometry.notna() & ~g.geometry.is_empty].copy().reset_index(drop=True)
    if g.crs is None:
        b, rb = g.total_bounds, tuple(BOUNDS)
        if not (b[2] >= rb[0] and b[0] <= rb[2] and b[3] >= rb[1] and b[1] <= rb[3]):
            raise ValueError(f"Cannot safely assign raster CRS to {path}")
        print("Missing CRS, assigning Baego raster CRS:", path.name)
        g = g.set_crs(CRS)
    else:
        g = g.to_crs(CRS)
    g["geometry"] = gpd.GeoSeries([polygonal(x) for x in g.geometry], index=g.index, crs=CRS)
    return g.loc[~g.geometry.is_empty].reset_index(drop=True)

rois = load_polygons(ROI_FILE)
assert len(rois) == 6, f"Expected exactly six ROIs, found {len(rois)}"
refs = load_polygons(KRISTOF_FILE)
refs = refs.loc[refs.geometry.area >= REF_MIN_AREA_M2].reset_index(drop=True)
pred = {(r,m):load_polygons(PRED_ROOT/f"roi{r}_{m}.gpkg")
        for r in ROI_IDS for m in METHODS}
print(f"Loaded {len(refs)} Kristof reference crowns >= {REF_MIN_AREA_M2} m²")
display(pd.DataFrame([{"ROI":r, "area_ha":rois.geometry.iloc[r].area/1e4,
                       "n_ref_centroid":int(refs.geometry.centroid.within(rois.geometry.iloc[r]).sum()),
                       **{f"{m}_halo":len(pred[r,m]) for m in METHODS}}
                      for r in ROI_IDS]))

## 03 — Canopy footprint agreement across all six ROIs

The union of **all reference crowns intersecting each ROI** is compared with the union of **all SAM polygons intersecting it**, including the 40 m halo. Both are clipped to the ROI. This gives spatially fair canopy coverage even for crowns centred outside the evaluation boundary.

**Precision:** common covered area / SAM-covered area. **Recall:** common covered area / reference-covered area. **IoU:** intersection / union. These are **area-based**, not individual-crown detection scores. Cover bias is expressed in absolute percentage points.

In [ ]:
# 03 — Area-based canopy agreement, with exact vector geometry
def clipped_union(gdf, region):
    if gdf.empty:
        return Polygon()
    candidates = gdf.loc[gdf.geometry.intersects(region)]
    parts = [polygonal(g.intersection(region)) for g in candidates.geometry]
    parts = [g for g in parts if not g.is_empty and g.area > 0]
    return polygonal(unary_union(parts)) if parts else Polygon()

footprints, area_rows = {}, []
for roi_id in ROI_IDS:
    region = polygonal(rois.geometry.iloc[roi_id])
    truth = clipped_union(refs, region)
    for method in METHODS:
        estimated = clipped_union(pred[roi_id, method], region)
        common = polygonal(truth.intersection(estimated))
        sam_only = polygonal(estimated.difference(truth))
        ref_only = polygonal(truth.difference(estimated))
        a_ref, a_sam = truth.area, estimated.area
        a_int = common.area
        a_union = a_ref + a_sam - a_int
        scale = 100 / region.area
        record = {
            "ROI": roi_id, "method": method, "roi_ha": region.area/1e4,
            "ref_cover_pct":a_ref*scale, "sam_cover_pct":a_sam*scale,
            "cover_bias_pp":(a_sam-a_ref)*scale,
            "agreement_pct_roi":a_int*scale,
            "sam_only_pct_roi":sam_only.area*scale,
            "reference_only_pct_roi":ref_only.area*scale,
            "canopy_IoU":a_int/a_union if a_union>0 else np.nan,
            "area_precision":a_int/a_sam if a_sam>0 else np.nan,
            "area_recall":a_int/a_ref if a_ref>0 else np.nan,
        }
        # The partition must fully reconstruct each canopy union.
        assert np.isclose(a_int+sam_only.area,a_sam,rtol=1e-7,atol=1e-4)
        assert np.isclose(a_int+ref_only.area,a_ref,rtol=1e-7,atol=1e-4)
        area_rows.append(record)
        footprints[(roi_id,method)] = dict(roi=region,ref=truth,sam=estimated,
                                           common=common,sam_only=sam_only,ref_only=ref_only)
AREA = pd.DataFrame(area_rows)
print("=== Canopy comparison: percentage points and spatial agreement ===")
display(AREA.round(3))
AREA.to_csv(OUT/"02_canopy_area_agreement.csv",index=False)
print("Saved:",OUT/"02_canopy_area_agreement.csv")

## 04 — Summary plots: ecological divergence and decomposition

Report per-ROI canopy overlap, omission and commission explicitly, rather than inferring agreement from cover percentages alone. A model could match cover percentage while placing polygons in the wrong locations.

In [ ]:
# 04 — All-six comparison and canopy error decomposition
fig, axes = plt.subplots(2, 1, figsize=(12, 9), constrained_layout=True)
for method, marker in (("author_style","o"),("nms_060","s")):
    subset = AREA.loc[AREA.method==method].sort_values("ROI")
    axes[0].plot(subset.ROI,subset.canopy_IoU,marker=marker,label=method)
axes[0].set(xlabel="Baego ROI",ylabel="Canopy footprint IoU",ylim=(0,1),
            title="Spatial canopy agreement (union footprint)")
axes[0].set_xticks(ROI_IDS); axes[0].legend();axes[0].grid(alpha=.2)
nms = AREA.loc[AREA.method=="nms_060"].sort_values("ROI")
x=np.arange(6); width=.36
axes[1].bar(x-width/2,nms.sam_only_pct_roi,width,label="SAM-only (% ROI)")
axes[1].bar(x+width/2,nms.reference_only_pct_roi,width,label="Reference-only (% ROI)")
axes[1].set(xlabel="Baego ROI",ylabel="Percent of ROI area",
            title="Distinct spatial canopy errors after NMS")
axes[1].set_xticks(x); axes[1].set_xticklabels(ROI_IDS);axes[1].legend()
fig.savefig(OUT/"02_canopy_summary.png",dpi=180,bbox_inches="tight")
plt.show()

## 05 — RGB maps: agreement, SAM-only and reference-only canopy

Map **ROIs 1, 3 and 4** at the same spatial footprint. For each ROI, show a natural RGB view and categorical canopy agreement map. Categories are mutually exclusive inside the ROI: **both**, **SAM-only**, **reference-only** and **neither**.

These maps use vector clipping and a rasterized display grid; **quantitative areas in Section 03 are computed from precise vectors**, not from plot pixels.

In [ ]:
# 05 — Categorical error maps, plotted over local RGB at raster resolution
from rasterio.features import rasterize
from matplotlib.colors import ListedColormap, BoundaryNorm

def show_one_roi(roi_id, method="nms_060"):
    fp=footprints[(roi_id,method)]
    region=fp["roi"]
    with rasterio.open(RGB) as ds:
        window=from_bounds(*region.bounds,transform=ds.transform).round_offsets().round_lengths()
        window=window.intersection(Window(0,0,ds.width,ds.height))
        # Use native pixels; these ROIs are modest in extent.
        image=ds.read([1,2,3],window=window).transpose(1,2,0).astype(float)
        trans=ds.window_transform(window)
        extent=(ds.window_bounds(window)[0],ds.window_bounds(window)[2],
                ds.window_bounds(window)[1],ds.window_bounds(window)[3])
    lo,hi=np.nanpercentile(image,[2,98])
    display_rgb=np.clip((image-lo)/max(hi-lo,1e-9),0,1)
    def raster_geom(g):
        if g.is_empty:return np.zeros(image.shape[:2],dtype=np.uint8)
        return rasterize([(g,1)],out_shape=image.shape[:2],transform=trans,fill=0,dtype=np.uint8)
    inside=raster_geom(region).astype(bool)
    t=raster_geom(fp["ref"]).astype(bool)
    p=raster_geom(fp["sam"]).astype(bool)
    classes=np.zeros(inside.shape,dtype=np.uint8)
    classes[inside & t & p]=1
    classes[inside & ~t & p]=2
    classes[inside & t & ~p]=3
    classes[~inside]=4
    cmap=ListedColormap(["#d1d1d1","#3e9060","#ea884e","#4e83cc","#ffffff"])
    fig,axs=plt.subplots(1,2,figsize=(13,6),constrained_layout=True)
    axs[0].imshow(display_rgb,extent=extent,origin="upper")
    axs[1].imshow(classes,extent=extent,origin="upper",cmap=cmap,vmin=-.5,vmax=4.5,
                  interpolation="nearest")
    for ax in axs:
        x,y=region.exterior.xy if region.geom_type=="Polygon" else ([],[])
        if len(x):ax.plot(x,y,color="black",linewidth=1.1)
        ax.set_aspect("equal");ax.set_xticks([]);ax.set_yticks([])
    axs[0].set_title(f"Baego ROI {roi_id}: RGB")
    axs[1].set_title(f"ROI {roi_id}: SAM1 {method} vs Kristof")
    legend=[Patch(facecolor=cmap.colors[i],label=label) for i,label in
            [(0,"Neither"),(1,"Both"),(2,"SAM-only"),(3,"Reference-only")]]
    axs[1].legend(handles=legend,loc="lower left",fontsize=8)
    fig.savefig(OUT/f"02_ROI{roi_id}_{method}_canopy_map.png",dpi=200,bbox_inches="tight")
    plt.show()

for roi_id in MAP_ROIS:
    show_one_roi(roi_id,"nms_060")

## 06 — Crown-association diagnostics (merging, fragmentation, omissions)

An independent **object-level analysis** based on **ROI-centroid-selected crowns**, using pairwise polygon IoU and an explicit spatial-association threshold.

- **One-to-one IoU ≥ 0.50**: Hungarian maximum-weight assignment, yielding a crown-level precision/recall/F1 proxy.
- **Potential merges**: a predicted crown significantly covers **at least two reference crowns** (each ≥25% of that reference polygon's area); this is a geometric flag, not a verified biological merger.
- **Potential splits**: one reference crown is significantly covered by at least two SAM polygons (each ≥25% of reference area).
- **Reference unmatched** and **SAM unmatched**: based on one-to-one IoU, not proof of missed trees or false detections.

Reference centroids must lie in the ROI. SAM centroids must also lie in the ROI for this object-level check. The separately computed canopy maps retain the halo-inclusive coverage. No model training or parameter search occurs.

In [ ]:
# 06 — Crown matching and candidate merge/split flags; six ROIs
def roi_centroid_geometries(gdf, region):
    return list(gdf.loc[gdf.geometry.centroid.within(region)].geometry)

def crown_audit(roi_id, method):
    region = rois.geometry.iloc[roi_id]
    truth = roi_centroid_geometries(refs,region)
    guess = roi_centroid_geometries(pred[roi_id,method],region)
    nr,npred=len(truth),len(guess)
    intersections=np.zeros((nr,npred),dtype=float)
    iou=np.zeros_like(intersections)
    for i,a in enumerate(truth):
        for j,b in enumerate(guess):
            if not a.intersects(b):continue
            shared=a.intersection(b).area
            intersections[i,j]=shared
            union=a.area+b.area-shared
            iou[i,j]=shared/union if union>0 else 0
    pairs=[]
    if nr and npred:
        # Assign only candidate matches above threshold; dummies permit unmatched crowns.
        score=np.where(iou>=MATCH_IOU,iou,0.)
        cost=np.zeros((nr+npred,nr+npred))
        cost[:nr,:npred]=-score
        aidx,bidx=linear_sum_assignment(cost)
        pairs=[(int(i),int(j),float(iou[i,j]))
               for i,j in zip(aidx,bidx) if i<nr and j<npred and iou[i,j]>=MATCH_IOU]
    nm=len(pairs)
    matched_ref={i for i,j,v in pairs}
    matched_pred={j for i,j,v in pairs}
    ref_area=np.array([a.area for a in truth])
    assoc=(intersections / np.maximum(ref_area[:,None],1e-12)) >= ASSOCIATION_REF_COVERAGE
    return {"ROI":roi_id,"method":method,
            "reference_n":nr,"predicted_n":npred,
            "matched_iou_ge_0_5":nm,
            "reference_unmatched":nr-len(matched_ref),
            "prediction_unmatched":npred-len(matched_pred),
            "crown_precision":nm/npred if npred else np.nan,
            "crown_recall":nm/nr if nr else np.nan,
            "crown_F1":2*nm/(nr+npred) if nr+npred else np.nan,
            "potential_merge_polygons":int(((assoc.sum(axis=0)>=2)).sum()) if npred else 0,
            "potential_split_references":int(((assoc.sum(axis=1)>=2)).sum()) if nr else 0}

CROWNS=pd.DataFrame([crown_audit(roi_id,method)
                    for roi_id in ROI_IDS for method in METHODS])
print("=== Object matching and geometric merge/split candidates ===")
display(CROWNS.round(3))
CROWNS.to_csv(OUT/"02_crown_association_summary.csv",index=False)
print("Saved:",OUT/"02_crown_association_summary.csv")

## 07 — Cross-metric interpretation and saved outputs

Interpret canopy performance **alongside** crown matching, not instead of it. In particular, similar canopy-cover percentages may hide spatial displacement; low crown F1 may coexist with substantial union-cover agreement.

The main results are `02_canopy_area_agreement.csv`, `02_crown_association_summary.csv`, `02_canopy_summary.png`, and individual RGB/error maps for ROIs 1, 3 and 4.

**Next scientific decision:** compare ROI 3 and ROI 4 canopy-only/reference-only fractions and candidate merge rates. If errors are driven by annotation incompleteness, revise the benchmark interpretation rather than tuning SAM indiscriminately.

In [ ]:
# 07 — Compact paired diagnostic report
summary = AREA.merge(CROWNS,on=["ROI","method"],validate="one_to_one")
cols=["ROI","method","cover_bias_pp","canopy_IoU","area_precision","area_recall",
      "sam_only_pct_roi","reference_only_pct_roi","crown_F1",
      "reference_unmatched","potential_merge_polygons","potential_split_references"]
display(summary.loc[:,cols].round(3))
summary.to_csv(OUT/"02_full_spatial_and_crown_diagnostics.csv",index=False)
print("Finished. Outputs:",OUT)
print("No original SAM1 predictions or human annotations were modified.")